# ChatGPMe Colab MVP

This notebook is the clean Colab path for the project. It uses the repo scripts directly instead of packaging a separate bundle.

It supports two setups:
- Repo already available in the Colab filesystem or through the VS Code Colab extension
- Repo stored in Google Drive and mounted into Colab


## 1. Configure paths

Set `REPO_DIR` to wherever this repo lives in Colab.

Examples:
- `/content/ChatGPMe`
- `/content/drive/MyDrive/ChatGPMe`


In [ ]:
%cd ..


anscombe.json*                mnist_test.csv
california_housing_test.csv   mnist_train_small.csv
california_housing_train.csv  README.md*


In [7]:
REPO_DIR = "/content/ChatGPMe"  # update if your repo lives elsewhere
GOOGLE_DRIVE_DOCS_DIR = f"{REPO_DIR}/google_drive_docs"
MVP_DATASET_PATH = f"{REPO_DIR}/data/processed_mvp/train_mvp.jsonl"
ADAPTER_DIR = f"{REPO_DIR}/artifacts/tinyllama-style-lora-colab"
AUTHOR = "Shakespeare"  # name used in the evaluation report


## 2. Optional: mount Google Drive

Run this only if the repo or documents are in Drive.

In [ ]:
from pathlib import Path

if not Path(REPO_DIR).exists():
    from google.colab import drive
    drive.mount('/content/drive')
    print('Mounted Drive. Update REPO_DIR above if needed, then rerun the notebook from the path cell.')
else:
    print(f'Repo found at {REPO_DIR}')


## 3. Check GPU

In [8]:
!nvidia-smi || true

Mon Mar 23 21:57:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   41C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 4. Install dependencies

In [9]:
%cd {REPO_DIR}
!python -m pip install -U pip
!python -m pip install torch transformers datasets peft accelerate sentencepiece

[Errno 2] No such file or directory: '/Users/Drew/ChatGPMe'
/content


## 5. Build corpus files from exported Google Docs

Skip this cell if `data/corpus` and `data/processed` already exist in the repo.

In [13]:
%cd {REPO_DIR}
!python scripts/extract_corpus.py --input-dir "{GOOGLE_DRIVE_DOCS_DIR}" --output-dir data/corpus --manifest data/corpus/manifest.json

[Errno 2] No such file or directory: '/Users/Drew/ChatGPMe'
/
python3: can't open file '//scripts/extract_corpus.py': [Errno 2] No such file or directory


## 6. Build chunked training data

In [ ]:
%cd {REPO_DIR}
!python scripts/build_lora_dataset.py --input-dir data/corpus --output-dir data/processed

## 7. Create an MVP subset

This keeps the first Colab run small and cheap. Increase it once the pipeline works.

In [ ]:
%cd {REPO_DIR}
!mkdir -p data/processed_mvp
!head -n 512 data/processed/train.jsonl > data/processed_mvp/train_mvp.jsonl
!wc -l data/processed_mvp/train_mvp.jsonl

## 8. Train the LoRA adapter

This is the main MVP training cell.

In [ ]:
%cd {REPO_DIR}
!python scripts/train_lora.py \
  --model-name TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
  --dataset-path "{MVP_DATASET_PATH}" \
  --output-dir "{ADAPTER_DIR}" \
  --epochs 1 \
  --max-steps 30 \
  --batch-size 2 \
  --grad-accum 4 \
  --max-length 512 \
  --save-steps 10

## 9. Run generation with the trained adapter

In [ ]:
%cd {REPO_DIR}
!python scripts/generate_with_adapter.py \
  --model-name TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
  --adapter-path "{ADAPTER_DIR}" \
  --prompt "Write a short reflective paragraph about building a class project with two teammates." \
  --max-new-tokens 80

## 10. Inspect artifacts

In [ ]:
%cd {REPO_DIR}
!find artifacts -maxdepth 2 -type f | sort

## 11. Evaluate: baseline vs adapter

Grades the adapter's outputs against the base model using Phi-3-mini as the judge — no API key needed.

Runs in two steps so GPU memory is never overloaded:
1. Generate outputs with TinyLlama (base + adapter)
2. Unload TinyLlama, load Phi-3-mini as judge, score everything

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)
from grader_colab import grade, print_report

In [ ]:
import torch, gc
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

EVAL_PROMPTS = [
    {"text": "Write about the passage of time and what it means to grow older.", "genre": "reflective"},
    {"text": "Describe the feeling of losing something important.", "genre": "emotional"},
    {"text": "Write about ambition: what drives people to want more.", "genre": "philosophical"},
    {"text": "Describe a place you remember clearly from your past.", "genre": "descriptive"},
    {"text": "Write about the relationship between power and responsibility.", "genre": "philosophical"},
    {"text": "Describe what it feels like to face an uncertain future.", "genre": "emotional"},
]

BASE_MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

def generate(model, prompt, max_new_tokens=80):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                              temperature=0.7, do_sample=True,
                              pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME, torch_dtype=torch.float16).to(device)
print("Generating baseline outputs...")
baselines = [{"text": generate(base_model, p["text"])} for p in EVAL_PROMPTS]

print("Loading adapter...")
adapter_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
print("Generating candidate outputs...")
candidates = [{"text": generate(adapter_model, p["text"])} for p in EVAL_PROMPTS]

del adapter_model, base_model, tokenizer
gc.collect()
torch.cuda.empty_cache()
print("Done. GPU memory freed.")


In [ ]:
JUDGE_MODEL = "microsoft/Phi-3-mini-4k-instruct"
print(f"Loading judge: {JUDGE_MODEL}")
judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL)
judge_model = AutoModelForCausalLM.from_pretrained(JUDGE_MODEL, torch_dtype=torch.float16).to(device)
print("Judge ready.")

In [ ]:
report = grade(
    author=AUTHOR,
    prompts=EVAL_PROMPTS,
    baselines=baselines,
    candidates=candidates,
    judge_model=judge_model,
    judge_tokenizer=judge_tokenizer,
)
print_report(report)

In [ ]:
from pathlib import Path
report_path = Path(ADAPTER_DIR) / "eval_report.json"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(report.to_json())
print(f"Report saved to {report_path}")